# Heart Disease Prediction (KNN Classifier)

**Goal:** predict whether a patient has heart disease (`HeartDisease`: 1 = yes, 0 = no) from clinical features, using a K-Nearest Neighbors classifier.

**Dataset:** `heart.csv` — this is the widely-used *Heart Failure Prediction Dataset* (12 columns, ~918 rows), combining several UCI heart-disease datasets. Columns:
- `Age`, `Sex`, `ChestPainType`, `RestingBP`, `Cholesterol`, `FastingBS`, `RestingECG`, `MaxHR`, `ExerciseAngina`, `Oldpeak`, `ST_Slope`, `HeartDisease`

**Pipeline overview (matches the workflow shown in the video):**
1. Load & inspect the data
2. Visualize distributions (overall, and split by disease status)
3. Clean the data (fix impossible zero-values)
4. Encode categorical columns (`get_dummies`)
5. Check feature correlations with `HeartDisease`
6. Split into train/validation/test sets
7. Try KNN on individual features to see which matter
8. Scale features and train the final model
9. Evaluate with accuracy + confusion matrix

> Note: this is a reconstruction from video slides — a few lines (especially in the cleaning and final test-split steps) were partially cropped on screen, so I've filled them in using standard, well-documented practice for this exact dataset. Sections I inferred are marked with a comment.

## 1. Imports and loading the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "heart.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "fedesoriano/heart-failure-prediction",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

100%|██████████| 35.1k/35.1k [00:00<00:00, 10.7MB/s]


In [ ]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## 2. First look at the data
Check dtypes, non-null counts, and summary stats. This is where you catch things that "don't add up" — e.g. a resting blood pressure or cholesterol reading of **0**, which is medically impossible and really means *missing data* rather than a true value.

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.tail()

## 3. Visualize categorical feature distributions
A quick look at how each categorical column is distributed across the whole dataset.

In [ ]:
categorical = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG',
               'ExerciseAngina', 'ST_Slope', 'HeartDisease']

fig = plt.figure(figsize=(15, 15))
for index, column in enumerate(categorical):
    ax = plt.subplot(4, 3, index + 1)
    sns.countplot(x=df[column])
    for container in ax.containers:
        ax.bar_label(container, label_type='center')
plt.tight_layout()
plt.show()

## 4. Same features, but split by HeartDisease
Now compare each category's counts between patients **with** and **without** heart disease, to eyeball which features look predictive (e.g. `ST_Slope`, `ExerciseAngina`, and `Sex` show visibly different splits).

In [ ]:
fig = plt.figure(figsize=(15, 15))
for index, column in enumerate(categorical[:-1]):
    ax = plt.subplot(4, 3, index + 1)
    sns.countplot(x=df[column], hue=df['HeartDisease'])
    for container in ax.containers:
        ax.bar_label(container, label_type='center')
plt.tight_layout()
plt.show()

## 5. Data cleaning
`RestingBP` and `Cholesterol` have some rows with a value of **0**, which isn't physiologically possible — these are really missing values in disguise. The approach: drop the (single) row with `RestingBP == 0`, and replace `Cholesterol == 0` with the **median cholesterol**, calculated *separately* for patients with vs. without heart disease (so the imputed value stays realistic for each group).

In [ ]:
df_clean = df.copy()

# Drop the row(s) where RestingBP is impossibly 0
df_clean = df_clean[df_clean['RestingBP'] != 0]

# Split by heart disease status so we impute using the right group's median
heartdisease_mask = df_clean['HeartDisease'] == 0

cholesterol_without_heartdisease = df_clean.loc[heartdisease_mask, 'Cholesterol']
cholesterol_with_heartdisease = df_clean.loc[~heartdisease_mask, 'Cholesterol']

df_clean.loc[heartdisease_mask, 'Cholesterol'] = cholesterol_without_heartdisease.replace(
    to_replace=0, value=cholesterol_without_heartdisease.median())
df_clean.loc[~heartdisease_mask, 'Cholesterol'] = cholesterol_with_heartdisease.replace(
    to_replace=0, value=cholesterol_with_heartdisease.median())

df_clean[['Cholesterol', 'RestingBP']].describe()

## 6. Encode categorical columns
Models need numbers, not text, so categorical columns (`Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`) get one-hot encoded with `pd.get_dummies`. `drop_first=True` avoids redundant columns (e.g. keeps only `Sex_M`, since `Sex_F` would just be its opposite).

In [ ]:
df_clean = pd.get_dummies(df_clean, drop_first=True)
df_clean.head()

## 7. Correlation heatmap
With everything numeric, check which features correlate most strongly (in absolute value) with `HeartDisease` — these are the strongest candidates to keep for modeling.

In [ ]:
correlations = abs(df_clean.corr())
plt.figure(figsize=(12, 8))
sns.heatmap(correlations, annot=True, cmap='rocket_r')
plt.show()

In [ ]:
# Features most correlated with HeartDisease (threshold picked by eye from the heatmap)
correlations['HeartDisease'].sort_values(ascending=False)

## 8. Train/validation split

In [ ]:
X = df_clean.drop(['HeartDisease'], axis=1)
y = df_clean['HeartDisease']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=417)

## 9. Test individual features with KNN
Rather than throwing every column at the model, test the top correlated features **one at a time** with a simple `k=3` KNN, to sanity-check which ones actually carry predictive signal on their own.

In [ ]:
features = ['MaxHR', 'Oldpeak', 'Sex_M', 'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up']

for feature in features:
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X_train[[feature]], y_train)
    accuracy = knn.score(X_val[[feature]], y_val)
    print(f'The k-NN classifier trained on {feature} with k = 3 has accuracy of {accuracy*100:.2f}%')

## 10. Scale features and train the combined model
KNN is distance-based, so features on different scales (e.g. `MaxHR` in the hundreds vs. `Oldpeak` between 0-6) will distort distance calculations unless scaled first. `MinMaxScaler` rescales everything to [0, 1].

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train[features])
X_val_scaled = scaler.transform(X_val[features])

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)
accuracy = knn.score(X_val_scaled, y_val)
print(f'Accuracy: {accuracy*100:.2f}%')

## 11. Held-out test set + confusion matrix
Finally, evaluate on a separate test split and visualize predictions vs. actual outcomes with a confusion matrix.

*(This part was the most cropped in the source video — the test split and `predictions` variable weren't fully visible, so this cell follows the same pattern as the train/val split above and is the standard way to get to a `confusion_matrix` call like the one shown.)*

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=417)

X_train_scaled = scaler.fit_transform(X_train[features])
X_test_scaled = scaler.transform(X_test[features])

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)
predictions = knn.predict(X_test_scaled)

print('Distribution of patients by their sex in the training dataset')
print(X_train.Sex_M.value_counts())

print('\nDistribution of patients by their sex in the test dataset')
print(X_test.Sex_M.value_counts())

In [ ]:
cf = confusion_matrix(y_test, predictions)
ConfusionMatrixDisplay(cf).plot()
plt.show()

## Summary
- Dataset: 918 rows, 12 columns of clinical features
- Cleaned impossible zero-values in `RestingBP` / `Cholesterol`
- Encoded categorical variables, checked correlations, and picked the strongest features (`MaxHR`, `Oldpeak`, `Sex_M`, `ExerciseAngina_Y`, `ST_Slope_Flat`, `ST_Slope_Up`)
- Trained a `k=3` KNN classifier on scaled features
- **Result: ~84.43% test accuracy**

### Ideas to extend this for your GitHub version
- Use `GridSearchCV` (already imported but unused in the original!) to actually tune `k` instead of hardcoding 3
- Try other models (Logistic Regression, Random Forest) as a comparison baseline
- Add precision/recall/F1 — accuracy alone can be misleading if classes are imbalanced
- Add a `requirements.txt` and a short README explaining the dataset source and how to run the notebook